# Kvasir Robust Segmentation — Heavy Real-Data Training
Trains the ResNet34 U-Net baseline and corruption-augmented EfficientNet-B0 DeepLabV3+ proposal across five deterministic folds. Outputs are uploaded to the requested Google Drive folder.

In [ ]:
import os, sys, subprocess, json, shutil, pathlib, torch, numpy as np
from PIL import Image
print({'python': sys.version, 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})

In [ ]:
!git clone -q https://github.com/RedHaider/haider-research-analysis.git
%cd /content/haider-research-analysis/kvasir-robust-segmentation-real-data
!pip -q install -e .[dev]

In [ ]:
!pip -q install datasets
from datasets import load_dataset
DATA_DIR = pathlib.Path('/content/data/kvasir-seg/Kvasir-SEG')
(DATA_DIR/'images').mkdir(parents=True, exist_ok=True)
(DATA_DIR/'masks').mkdir(parents=True, exist_ok=True)
hf = load_dataset('MedOtter/KvasirSEG')
for split in hf.values():
    for row in split:
        name = row['image_id'] + '.png'
        row['image'].convert('RGB').save(DATA_DIR/'images'/name)
        mask = np.asarray(row['mask'].convert('L'))
        mask = ((mask > 0).astype(np.uint8) * 255)
        Image.fromarray(mask, mode='L').save(DATA_DIR/'masks'/name)
images = list((DATA_DIR/'images').glob('*'))
masks = list((DATA_DIR/'masks').glob('*'))
assert len(images) == len(masks) == 1000, (len(images), len(masks))
foreground = [float((np.asarray(Image.open(p)) > 127).mean()) for p in masks]
assert min(foreground) > 0 and max(foreground) < 1, (min(foreground), max(foreground))
print('Verified:', len(images), 'images and', len(masks), 'masks')

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/heavy.yaml'))
cfg['data_root'] = '/content/data/kvasir-seg/Kvasir-SEG'
cfg['output_root'] = '/content/kvasir-results'
cfg['num_workers'] = 2
with open('/content/heavy_colab.yaml','w') as f: yaml.safe_dump(cfg,f)
print(cfg)

In [ ]:
# Full heavy protocol: 10 training runs. Change FOLDS to [0] for a quick pilot.
FOLDS = list(range(5))
for fold in FOLDS:
    for model in ('baseline', 'proposed'):
        print(f'\n=== {model} fold {fold} ===')
        subprocess.run([sys.executable, '-m', 'kvasir_robust.train', '--config', '/content/heavy_colab.yaml', '--fold', str(fold), '--model', model], check=True)

In [ ]:
# Upload checkpoints, histories and metrics to the exact shared Drive folder.
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
drive = build('drive', 'v3')
PARENT_FOLDER_ID = '14m8Q9LjaCK2MV_irrBNxlUO9drXIINpM'
folder = drive.files().create(body={'name':'kvasir-robust-segmentation-results','mimeType':'application/vnd.google-apps.folder','parents':[PARENT_FOLDER_ID]}, fields='id').execute()
for path in pathlib.Path('/content/kvasir-results').rglob('*'):
    if path.is_file():
        rel = str(path.relative_to('/content/kvasir-results')).replace(os.sep, '__')
        drive.files().create(body={'name':rel,'parents':[folder['id']]}, media_body=MediaFileUpload(str(path), resumable=True), fields='id').execute()
print('Uploaded results folder ID:', folder['id'])